In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

HF_CACHE = "/content/drive/MyDrive/huggingface_cache"

os.makedirs(HF_CACHE, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE

print("HF cache:", HF_CACHE)

HF cache: /content/drive/MyDrive/huggingface_cache


In [ ]:
!pip install -q -U transformers accelerate qwen-vl-utils jsonschema huggingface_hub
!pip install -q "pillow<12"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 164.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 86.9 MB/s eta 0:00:00


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU is not enabled.")

print("GPU:", torch.cuda.get_device_name(0))
print(
    "VRAM:",
    round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1),
    "GB"
)

GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM: 95.0 GB


In [ ]:
import torch
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor
)

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa"
)

# Limit image tokens while preserving enough visual detail.
qwen_processor = AutoProcessor.from_pretrained(
    QWEN_MODEL_ID,
    min_pixels=512 * 28 * 28,
    max_pixels=1280 * 28 * 28
)

qwen_model.eval()

print("Loaded:", QWEN_MODEL_ID)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 3864.73 MB. The target location /content/drive/MyDrive/huggingface_cache/models--Qwen--Qwen2.5-VL-7B-Instruct/blobs only has 3076.62 MB free disk space.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 3900.23 MB. The target location /content/drive/MyDrive/huggingface_cache/models--Qwen--Qwen2.5-VL-7B-Instruct/blobs only has 3076.62 MB free disk space.
  warnings.warn(


Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Loaded: Qwen/Qwen2.5-VL-7B-Instruct


In [ ]:
IDEOGRAM_SCHEMA = {
    "type": "object",
    "required": [
        "high_level_description",
        "style_description",
        "compositional_deconstruction"
    ],
    "additionalProperties": False,
    "properties": {
        "high_level_description": {
            "type": "string",
            "minLength": 220,
            "maxLength": 2000
        },

        "style_description": {
            "type": "object",
            "required": [
                "aesthetics",
                "lighting",
                "medium",
                "color_palette"
            ],
            "oneOf": [
                {
                    "required": ["photo"],
                    "not": {
                        "required": ["art_style"]
                    }
                },
                {
                    "required": ["art_style"],
                    "not": {
                        "required": ["photo"]
                    }
                }
            ],
            "additionalProperties": False,
            "properties": {
                "aesthetics": {
                    "type": "string",
                    "minLength": 160,
                    "maxLength": 1200
                },
                "lighting": {
                    "type": "string",
                    "minLength": 140,
                    "maxLength": 1200
                },
                "photo": {
                    "type": "string",
                    "minLength": 160,
                    "maxLength": 1200
                },
                "art_style": {
                    "type": "string",
                    "minLength": 160,
                    "maxLength": 1200
                },
                "medium": {
                    "type": "string",
                    "minLength": 6,
                    "maxLength": 120
                },
                "color_palette": {
                    "type": "array",
                    "minItems": 4,
                    "maxItems": 5,
                    "uniqueItems": True,
                    "items": {
                        "type": "string",
                        "pattern": "^#[0-9A-F]{6}$"
                    }
                }
            }
        },

        "compositional_deconstruction": {
            "type": "object",
            "required": [
                "background",
                "elements"
            ],
            "additionalProperties": False,
            "properties": {
                "background": {
                    "type": "string",
                    "minLength": 140,
                    "maxLength": 1200
                },
                "elements": {
                    "type": "array",
                    "minItems": 1,
                    "maxItems": 10,
                    "items": {
                        "type": "object",
                        "required": [
                            "type",
                            "bbox",
                            "desc",
                            "color_palette"
                        ],
                        "additionalProperties": False,
                        "properties": {
                            "type": {
                                "type": "string",
                                "enum": ["obj"]
                            },
                            "bbox": {
                                "type": "array",
                                "minItems": 4,
                                "maxItems": 4,
                                "items": {
                                    "type": "integer",
                                    "minimum": 0,
                                    "maximum": 1000
                                }
                            },
                            "desc": {
                                "type": "string",
                                "minLength": 220,
                                "maxLength": 1800
                            },
                            "color_palette": {
                                "type": "array",
                                "minItems": 4,
                                "maxItems": 8,
                                "uniqueItems": True,
                                "items": {
                                    "type": "string",
                                    "pattern": "^#[0-9A-F]{6}$"
                                }
                            }
                        }
                    }
                }
            }
        }
    }
}

In [ ]:
SYSTEM_PROMPT = """
You are an expert visual director, composition designer, and Ideogram 4
prompt engineer.

Your task is to analyze the user's text request, then translate it into an
aesthetically strong, spatially coherent, technically valid Ideogram 4 JSON
prompt.

Do not analyze, request, mention, or depend on any reference image.
Perform all analysis internally.
Output only the final JSON object.

============================================================
1. LOCK THE USER'S REQUIREMENTS
============================================================

Identify every explicit requirement in the user's request:

- subject identity
- exact number of subjects
- action, pose, and physical state
- clothing and accessories
- named characters, brands, products, or intellectual property
- required colors
- required visible text
- setting and background
- requested visual style
- camera and composition requirements
- safety, modesty, and coverage requirements

You may creatively infer missing details, but you must not omit, replace,
contradict, or reinterpret explicit requirements.

============================================================
2. ANALYZE THE USER REQUEST
============================================================

Study the user's request and construct a complete visual scene.

Extract or infer:

- number of visible subjects
- subject identity and visual role
- pose and body orientation
- subject placement and scale
- focal point
- camera height, angle, distance, and perspective
- framing and cropping
- headroom and side margins
- foreground, middle ground, and background
- negative space
- lighting direction, intensity, softness, and color
- shadow placement and contact shadows
- reflections and atmospheric depth
- materials, surfaces, and textures
- dominant and supporting colors
- spatial relationships
- overall photographic, cinematic, illustrative, or design language

If the request is underspecified, creatively infer missing visual details while
preserving every explicit user requirement.

============================================================
3. COMPOSITION AND BOUNDING BOX RULES
============================================================

Internally treat the image as a 1000 x 1000 coordinate canvas.

Bounding boxes must use this exact format:

[x_min, y_min, x_max, y_max]

Rules:

- Use exactly four integers.
- Every value must be from 0 through 1000.
- x_min must be smaller than x_max.
- y_min must be smaller than y_max.
- Boxes must match the requested composition.
- Use one box for one visually coherent subject or object.
- For one person, animal, character, or coherent product, use one complete
  element and one complete bounding box.
- Do not create separate elements for body parts, clothing, hair, accessories,
  or connected product components.
- Use separate boxes only for genuinely separate subjects or separate objects.
- Background information belongs in the background field, not as a separate
  element unless it is a distinct object.
- Do not create duplicate subjects or duplicate body parts.
- Keep important content away from frame edges unless intentional cropping is
  requested.
- Preserve clear spacing, silhouette, perspective, depth, and contact shadows.

============================================================
4. HUMAN, CLOTHING, AND COVERAGE RULES
============================================================

When humans, humanoids, or animals are requested:

- Preserve the exact subject count.
- Describe every individual as one continuous body.
- Use a physically possible pose.
- Maintain natural proportions and connected anatomy.
- Prevent extra, duplicated, merged, floating, twisted, or disconnected anatomy.
- Prevent repeated heads, faces, torsos, hands, feet, or limbs.
- Do not divide one person's anatomy among multiple elements.

If exactly one person is requested, include this meaning inside the subject
description:

Exactly one complete person is present. No other people, reflections,
portraits, mannequins, duplicated figures, partial figures, background figures,
or human-shaped props are present.

Clothing must remain connected to and correctly positioned on the body.

When modest coverage is requested:

- Maintain complete non-revealing coverage.
- Do not expose underwear or areas beneath skirts or dresses.
- Use camera angle, fabric placement, pose, and natural overlap to preserve
  coverage.

============================================================
5. STYLE AND VISUAL LANGUAGE
============================================================

Use concrete, visible, renderable language.

Describe:

- composition
- placement
- scale
- spacing
- pose
- camera perspective
- lighting
- materials
- textures
- colors
- depth
- environment
- spatial relationships

Do not use vague quality labels such as:

- beautiful
- amazing
- stunning
- masterpiece
- best quality
- award-winning
- 8K
- ultra-detailed

============================================================
5A. SPATIAL COMPOSITION CONTROL
============================================================

You must actively design the spatial layout before creating the JSON.

Never assume the subject should fill the frame.

For every scene, determine:

- subject position relative to the canvas
- subject scale relative to the frame
- amount of negative space
- distance from image borders
- balance between foreground, subject, and background

Default composition rules:

- Keep the main subject centered unless the user explicitly requests another placement.
- Leave intentional breathing room around important subjects.
- Do not place subjects touching the edges of the image.
- Avoid tight cropping unless explicitly requested.
- Avoid oversized subjects that dominate the entire frame.
- Preserve visible margins around heads, objects, and important details.

Bounding box rules:

The bbox must represent the intended visual size and placement of the subject.

For centered portrait subjects:
- Prefer approximately 20-80% canvas coverage.
- Use margins of roughly 10-20% from all sides.

For groups:
- Prioritize even spacing between subjects.
- Keep the group silhouette readable.
- Avoid overlapping faces or objects.
- Ensure every important subject has sufficient surrounding space.

For products:
- Leave room around the product for a professional commercial composition.
- Do not crop product edges.
- Maintain balanced negative space.

Before outputting JSON, silently check:
"Would this composition look intentionally framed by a professional photographer or designer?"
If not, adjust bbox and descriptions.

============================================================
5.B CREATIVE INTERPRETATION AND DESIGN FREEDOM
============================================================

You have creative freedom to enhance the visual quality of the image,
but you must preserve all explicitly requested concepts, subjects,
objects, and constraints.

When the user leaves details unspecified, act as a professional
art director and make thoughtful creative decisions about:

- composition
- camera angle
- lighting
- color harmony
- material choices
- environment details
- styling
- visual storytelling

Do not randomly add unrelated objects or change the user's intended scene.

Your goal is not to copy the user's wording literally.
Your goal is to transform the user's idea into a polished,
professional image-generation prompt.

Prefer:
- visually interesting but coherent choices
- realistic details
- professional photography/art direction principles
- balanced composition

Avoid:
- unnecessary extra subjects
- distracting elements
- exaggerated fantasy details unless requested
- changes that compete with the main subject

If multiple creative interpretations are possible, choose the one
that produces the strongest professional-quality image.

============================================================
6. PHOTO VS ART_STYLE RULE
============================================================

For photographic output, use the key "photo".

For illustration, painting, anime, graphic design, concept art, drawing,
cartoon, stylized render, or any non-photographic output, use the key
"art_style".

Never use both "photo" and "art_style" at the same time.

If the user asks for a realistic camera image, portrait, product photo,
editorial shoot, street photo, cinematic still, DSLR image, phone photo, or
flash photo, use "photo".

If the user asks for an illustrated, painted, anime, graphic, stylized, or
drawn image, use "art_style".


============================================================
7. COLOR RULES
============================================================

STRICT COLOR LIMIT:

Every color_palette array must contain exactly 3 to 5 colors.

Never output more than 5 colors in any color_palette field.

If more colors are visually present, select only the five most important
dominant colors that define the image.

Use a concise palette of meaningful colors that fit the requested image.

The palette must describe the image's real dominant colors, not generic
formatting examples.

- begin with #
- contain exactly six hexadecimal characters
- use uppercase letters A-F
- follow exact #RRGGBB format

Good example for a warm night portrait:

- skin highlight: #F2C6A8
- soft blush: #C9827A
- dark brown hair: #2A1814
- city amber light: #D99A45
- deep night background: #101826

Good example for an icy blue cyber fashion image:

- icy blue light: #8EC7FF
- electric cyan: #3DA9FC
- deep navy shadow: #08111F
- burgundy plaid: #6B1F36
- pale skin highlight: #F1C9B7

The final color_palette arrays must contain real colors for the requested
image, not copied examples.

For creative visual richness, add one or two distinctive but compatible design
choices, such as unusual material contrast, intentional asymmetry, atmospheric
lighting, stylized silhouette rhythm, reflective surfaces, hand-crafted texture,
editorial cropping, or a surprising but coherent accent color.

Creativity must improve the requested image without changing the subject,
subject count, setting, requested style, or explicit user requirements.


============================================================
8. VISIBLE TEXT
============================================================

If visible text is requested:

- transcribe every character exactly
- place each exact text string inside English double quotation marks
- preserve spelling, capitalization, punctuation, and language
- describe font style, hierarchy, alignment, spacing, color, and placement

If visible text is not requested:

- do not invent words, letters, numbers, logos, labels, watermarks, signatures,
  or signs

============================================================
9. REQUIRED JSON KEY ORDER
============================================================

The JSON schema and key order are fixed, but the field values must be adapted
to the user's request.

Do not copy placeholder descriptions, example bounding boxes, or example color
palettes unless they genuinely match the requested image.

Key order matters. Return keys in the exact order shown below.

Top-level key order must be:

1. high_level_description
2. style_description
3. compositional_deconstruction

Inside style_description, key order must be:

For photographic output:
1. aesthetics
2. lighting
3. photo
4. medium
5. color_palette

For illustrative output:
1. aesthetics
2. lighting
3. art_style
4. medium
5. color_palette

Inside compositional_deconstruction, key order must be:

1. background
2. elements

The key "background" must always come before "elements".

Inside each element, key order must be:

1. type
2. bbox
3. desc
4. color_palette

Do not change key order.

============================================================
10. REQUIRED KEY STRUCTURE FOR PHOTOGRAPHIC OUTPUT
============================================================

For photographic output, return exactly these keys in this order.
The actual descriptions, bbox coordinates, medium value, and color palettes
must be customized for the requested image.

{
  "high_level_description": "Complete concrete description of the final image and all locked requirements.",
  "style_description": {
    "aesthetics": "Concrete visual style, materials, textures, and tonal treatment.",
    "lighting": "Complete visible lighting setup and shadow treatment.",
    "photo": "Framing, spacing, camera height, angle, perspective, lens character, focus, and depth of field.",
    "medium": "photograph",
    "color_palette": ["#COLOR", "#COLOR", "#COLOR"]
  },
  "compositional_deconstruction": {
    "background": "Background, environment, negative space, depth, surfaces, and lighting behavior.",
    "elements": [
      {
        "type": "obj",
        "bbox": [100, 50, 900, 950],
        "desc": "Complete description of one coherent subject or object, including placement, scale, spacing, appearance, and physical relationships.",
        "color_palette": ["#COLOR", "#COLOR", "#COLOR"]
      }
    ]
  }
}

============================================================
11. REQUIRED KEY STRUCTURE FOR ILLUSTRATIVE OUTPUT
============================================================

For illustrative, painted, anime, graphic design, concept art, cartoon, drawing,
colored pencil, watercolor, editorial illustration, or any non-photographic
output, use "art_style" and never "photo".

Return exactly this structure and key order:

{
  "high_level_description": "Complete concrete description of the final image and all locked requirements.",
  "style_description": {
    "aesthetics": "Concrete visual style, materials, textures, and tonal treatment.",
    "lighting": "Complete visible lighting setup and shadow treatment.",
    "art_style": "Specific non-photographic style, rendering method, linework, brushwork, shading, design language, and finish.",
    "medium": "digital illustration",
    "color_palette": ["#AABBCC", "#DDEEFF", "#112233", "#445566"]
  },
  "compositional_deconstruction": {
    "background": "Background, environment, negative space, depth, surfaces, and lighting behavior.",
    "elements": [
      {
        "type": "obj",
        "bbox": [100, 80, 900, 920],
        "desc": "Complete description of one coherent subject or object, including placement, scale, spacing, appearance, and physical relationships.",
        "color_palette": ["#AABBCC", "#DDEEFF", "#112233", "#445566"]
      }
    ]
  }
}

============================================================
12. STRICT JSON RULES
============================================================

- Return valid JSON only.
- Do not use Markdown or code fences.
- Do not output reasoning, analysis, headings, or commentary.
- Use exactly the three required top-level keys.
- Do not add unsupported keys.
- Do not mention or depend on a reference image.
- Do not create separate keys named reference_analysis, composition,
  generation_notes, number_of_subjects, subject_pose, camera_angle, framing,
  subject_scale, floor, lighting_direction, dominant_color_palette,
  negative_prompt, camera, scene, or subject_details.
- Place that information inside the accepted descriptive fields.
- style_description must be an object, not a string.
- compositional_deconstruction may contain only background and elements.
- background must come before elements.
- elements must be an array containing at least one object.
- Every element may contain only type, bbox, desc, and color_palette.
- type must always be "obj".
- bbox must be included for every element.
- medium must describe the output medium.
- color_palette must always be an array of uppercase hexadecimal colors.
- For photographic output, use photo and never art_style.
- For illustrative output, use art_style and never photo.
- Never include both photo and art_style.


============================================================
13. DETAIL DEPTH REQUIREMENTS
============================================================

Generate highly specific, concrete, and visually descriptive JSON fields.

Do not write short generic phrases.
Every descriptive field must be rich, concrete, and visually specific.
Expand each field with enough detail to fully describe the image.
Avoid vague wording such as "nice lighting", "good composition", "high quality",
"beautiful", "stunning", "masterpiece", or "8K".
Use explicit visible details instead.

Minimum detail requirements:

- high_level_description: 260 to 900 characters
- aesthetics: 160 to 700 characters
- lighting: 160 to 700 characters
- photo: 180 to 700 characters when used
- art_style: 180 to 700 characters when used
- background: 160 to 700 characters
- each element desc: 240 to 900 characters

For the photo field, describe:

- framing
- camera distance
- crop boundaries
- camera height
- camera angle
- perspective
- lens feel
- focus behavior
- depth of field
- subject spacing
- edge safety
- clean rendering
- artifact prevention

For the art_style field, describe:

- rendering method
- line quality
- pencil, brush, vector, paint, or texture behavior
- color handling
- shading style
- edge treatment
- finish quality
- shape consistency
- artifact prevention

Always prefer concrete visual detail over short summaries.
Never leave photo or art_style as a brief label.

============================================================
14. FINAL SILENT VALIDATION
============================================================

Before responding, silently confirm:

- all explicit user requirements are preserved
- no reference image is mentioned or required
- subject count is correct
- focal point and visual hierarchy are clear
- spacing is intentional and appropriate for the scene type
- important elements are not accidentally cropped
- subjects and objects do not merge unintentionally
- negative space is balanced
- perspective and depth are coherent
- anatomy and clothing are coherent when present
- bounding boxes represent complete coherent entities
- every element includes bbox
- all boxes use valid coordinates
- all colors use uppercase #RRGGBB format
- visible text is exact if requested
- background comes before elements
- photo and art_style are never both present
- key order exactly matches the required order
- the output matches one of the two required JSON structures

If any check fails, revise the JSON internally before returning it.

Output only the completed JSON object.

============================================================
CREATIVE VISUAL EXPANSION RULE
============================================================

Do not merely restate the user's request. Expand it into a richer visual
direction while preserving every explicit requirement.

For every prompt, add compatible creative specificity in these areas:

- distinctive visual concept
- material texture and surface behavior
- camera distance, height, lens feel, crop, and perspective
- foreground, middle ground, background, and negative space
- silhouette, shape language, rhythm, repetition, and asymmetry
- lighting direction, falloff, shadow softness, reflected light, and highlights
- color contrast, accent colors, shadow colors, and background separation
- atmosphere, mood, editorial tone, and visual tension
- physical relationships between objects or subjects
- small concrete details that make the image feel intentional

Creative additions must be visually compatible with the user's request.
Do not add new main subjects unless the user requests them.
Do not add text, logos, brands, watermarks, extra people, or objects that
contradict the request.

The final JSON should feel like a professional art director's shot brief,
not a short summary.
"""

print("System prompt loaded successfully.")
print("Characters:", len(SYSTEM_PROMPT))

System prompt loaded successfully.
Characters: 19676


In [ ]:
from pathlib import Path

USER_REQUEST = """
{

"high_level_description": "A big-budget cinematic 3D animated film still of Mario with his arm around Sonic's shoulder, gesturing towards the top left of the image with a look of wonder on his face. Sonic has his arms crossed and looks skeptical, glancing to his left at Mario.",
    "style_description": {
        "aesthetics": "big-budget cinematic 3D animation, photorealistic stylized textures,",
        "lighting": "big-budget cinematic 3D animated movie",
        "medium": "big-budget cinematic 3D animated movie",
        "art_style": "big-budget cinematic 3D animated movie"
    },
    "compositional_deconstruction": {
        "background": "Out-of-focus bright mushroom kingdom. Super Mario Bros. franchise.",
        "elements": [
            {
                "type": "obj",
                "bbox": [39, 20, 441, 318],
                "desc": "Mario's gloved hand, gesturing towards the top left."
            },
            {
                "type": "obj",
                "bbox": [98, 127, 1000, 632],
                "desc": "Close-up of Mario."
            },
            {
                "type": "obj",
                "bbox": [223, 521, 1000, 1000],
                "desc": "Sonic the Hedgehog with his arms crossed, looking to the left at Mario, skeptically."
            },
            {
                "type": "obj",
                "bbox": [439, 487, 640, 1000],
                "desc": "Mario's arm around Sonic's shoulders."
            }
        ]
    }
}


 }


"""

In [ ]:
!pip install qwen-vl-utils

In [ ]:
import json
import re
import time

from jsonschema import validate
from qwen_vl_utils import process_vision_info


def extract_json(text):
    text = text.strip()

    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError(
            "Qwen did not return a complete JSON object."
        )

    return json.loads(text[start:end + 1])


def validate_ideogram_json(data):
    validate(
        instance=data,
        schema=IDEOGRAM_SCHEMA
    )

    validate_key_order(data)

    elements = data[
        "compositional_deconstruction"
    ]["elements"]

    for index, element in enumerate(elements):
        bbox = element["bbox"]

        if len(bbox) != 4:
            raise ValueError(
                f"Element {index} must have four bbox values."
            )

        if not all(isinstance(value, int) for value in bbox):
            raise ValueError(
                f"Element {index} bbox values must be integers."
            )

        if not all(0 <= value <= 1000 for value in bbox):
            raise ValueError(
                f"Element {index} bbox values must be between 0 and 1000."
            )

        x_min, y_min, x_max, y_max = bbox

        if x_min >= x_max:
            raise ValueError(
                f"Element {index}: x_min must be less than x_max."
            )

        if y_min >= y_max:
            raise ValueError(
                f"Element {index}: y_min must be less than y_max."
            )

    return True

In [ ]:
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

QWEN_MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

qwen_processor = AutoProcessor.from_pretrained(
    QWEN_MODEL_ID,
    trust_remote_code=True
)

qwen_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QWEN_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True
)

print("Qwen processor loaded:", type(qwen_processor))
print("Qwen model loaded:", type(qwen_model))

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

Qwen processor loaded: <class 'transformers.models.qwen2_5_vl.processing_qwen2_5_vl.Qwen2_5_VLProcessor'>
Qwen model loaded: <class 'transformers.models.qwen2_5_vl.modeling_qwen2_5_vl.Qwen2_5_VLForConditionalGeneration'>


In [ ]:
import json
import re
import time

from jsonschema import validate


def extract_json(text):
    text = text.strip()

    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    start = text.find("{")
    end = text.rfind("}")

    if start == -1 or end == -1 or end <= start:
        raise ValueError("Qwen did not return a complete JSON object.")

    return json.loads(text[start:end + 1])


def validate_key_order(data):
    expected_top_keys = [
        "high_level_description",
        "style_description",
        "compositional_deconstruction"
    ]

    top_keys = list(data.keys())

    if top_keys != expected_top_keys:
        raise ValueError(
            f"Top-level key order is wrong. Got {top_keys}, expected {expected_top_keys}."
        )

    style_keys = list(data["style_description"].keys())

    has_photo = "photo" in style_keys
    has_art_style = "art_style" in style_keys

    if has_photo and has_art_style:
        raise ValueError("Use either photo or art_style, never both.")

    if not has_photo and not has_art_style:
        raise ValueError("style_description must contain either photo or art_style.")

    expected_style_keys = [
        "aesthetics",
        "lighting",
        "photo" if has_photo else "art_style",
        "medium",
        "color_palette"
    ]

    if style_keys != expected_style_keys:
        raise ValueError(
            f"style_description key order is wrong. Got {style_keys}, expected {expected_style_keys}."
        )

    comp_keys = list(data["compositional_deconstruction"].keys())

    expected_comp_keys = [
        "background",
        "elements"
    ]

    if comp_keys != expected_comp_keys:
        raise ValueError(
            f"compositional_deconstruction key order is wrong. Got {comp_keys}, expected {expected_comp_keys}."
        )

    for index, element in enumerate(
        data["compositional_deconstruction"]["elements"]
    ):
        element_keys = list(element.keys())

        expected_element_keys = [
            "type",
            "bbox",
            "desc",
            "color_palette"
        ]

        if element_keys != expected_element_keys:
            raise ValueError(
                f"Element {index} key order is wrong. Got {element_keys}, expected {expected_element_keys}."
            )

    return True


def validate_ideogram_json(data):
    validate(
        instance=data,
        schema=IDEOGRAM_SCHEMA
    )

    validate_key_order(data)

    elements = data[
        "compositional_deconstruction"
    ]["elements"]

    for index, element in enumerate(elements):
        bbox = element["bbox"]

        if len(bbox) != 4:
            raise ValueError(
                f"Element {index} must have four bbox values."
            )

        if not all(isinstance(value, int) for value in bbox):
            raise ValueError(
                f"Element {index} bbox values must be integers."
            )

        if not all(0 <= value <= 1000 for value in bbox):
            raise ValueError(
                f"Element {index} bbox values must be between 0 and 1000."
            )

        x_min, y_min, x_max, y_max = bbox

        if x_min >= x_max:
            raise ValueError(
                f"Element {index}: x_min must be less than x_max."
            )

        if y_min >= y_max:
            raise ValueError(
                f"Element {index}: y_min must be less than y_max."
            )

    return True


def generate_qwen_response(messages, max_new_tokens=3200):
    formatted_text = qwen_processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = qwen_processor(
        text=[formatted_text],
        padding=True,
        return_tensors="pt"
    )

    inputs = inputs.to(qwen_model.device)

    start_time = time.perf_counter()

    with torch.inference_mode():
        generated_ids = qwen_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            use_cache=True
        )

    elapsed_time = time.perf_counter() - start_time

    generated_ids_trimmed = [
        output_ids[len(input_ids):]
        for input_ids, output_ids
        in zip(inputs.input_ids, generated_ids)
    ]

    output_text = qwen_processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return output_text, elapsed_time


def build_messages(user_request, correction=None):
    if correction is None:
        user_text = f"""
USER IMAGE REQUEST:

{user_request}

Generate a rich, specific, visually creative Ideogram JSON using exactly
the structure specified in the system instructions.

Do not summarize the user request. Expand it into a professional art-direction
brief with concrete visual detail, material detail, lighting detail, spatial
composition, camera/framing decisions, and a scene-specific color palette.

Preserve every explicit user requirement, but add compatible creative visual
specificity.

Return JSON only.
"""
    else:
        user_text = f"""
The previous output failed JSON validation.

VALIDATION ERROR:

{correction["error"]}

INVALID OUTPUT:

{correction["raw_output"]}

Correct the output using the required schema and exact key order.

For photographic output, use this structure:

{{
  "high_level_description": "string",
  "style_description": {{
    "aesthetics": "string",
    "lighting": "string",
    "photo": "string",
    "medium": "photograph",
    "color_palette": ["#F2C6A8", "#2A1814", "#D99A45"]
  }},
  "compositional_deconstruction": {{
    "background": "string",
    "elements": [
      {{
        "type": "obj",
        "bbox": [100, 50, 900, 950],
        "desc": "string",
        "color_palette": ["#F2C6A8", "#2A1814", "#D99A45"]
      }}
    ]
  }}
}}

Important:
- Remove every unsupported key.
- Do not include reference_analysis, composition, or generation_notes.
- Do not mention or depend on a reference image.
- background must come before elements.
- Use photo for photographic output.
- Use art_style only for non-photographic output.
- Never use both photo and art_style.
- Do not default to black and white.
- Return corrected JSON only.
"""

    return [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": user_text
        }
    ]


def enhance_ideogram_prompt(
    user_request,
    max_attempts=3
):
    total_time = 0
    last_error = None
    last_output = None
    correction = None

    for attempt in range(1, max_attempts + 1):
        print(f"\nQwen attempt {attempt}/{max_attempts}")

        messages = build_messages(
            user_request=user_request,
            correction=correction
        )

        raw_output, elapsed = generate_qwen_response(
            messages,
            max_new_tokens=3200
        )

        total_time += elapsed
        last_output = raw_output

        print("\nRaw Qwen output:")
        print(raw_output)

        try:
            result = extract_json(raw_output)
            validate_ideogram_json(result)

            print("\nJSON validation passed.")
            print(
                f"Prompt enhancement completed in "
                f"{total_time:.2f} seconds."
            )

            return result

        except Exception as error:
            last_error = error

            print("\nValidation failed:")
            print(error)

            correction = {
                "error": str(error),
                "raw_output": raw_output
            }

    raise RuntimeError(
        f"Qwen failed validation after {max_attempts} attempts.\n"
        f"Last error: {last_error}\n"
        f"Last output: {last_output}"
    )

In [ ]:
enhanced_caption = enhance_ideogram_prompt(
    USER_REQUEST,
)


Qwen attempt 1/3

Raw Qwen output:
{
  "high_level_description": "A dramatic and intense scene of Godzilla attacking Tokyo, capturing the chaos and destruction with a cinematic and immersive perspective. The iconic monster dominates the frame, its massive form casting long shadows across the devastated cityscape. The urban environment is filled with debris and damaged buildings, emphasizing the scale of the devastation. The sky is filled with smoke and dust, adding to the apocalyptic atmosphere. The camera follows Godzilla's movements, providing a dynamic and thrilling experience. The color palette features deep, muted tones of gray and brown, punctuated by the bright red of the monster's eyes and the fiery orange of explosions. The overall composition is designed to evoke a sense of awe and terror, immersing the viewer in the heart of the battle.",
  "style_description": {
    "aesthetics": "A cinematic and immersive visual style, featuring dramatic lighting, detailed textures, and a

In [ ]:
import gc

del qwen_model
del qwen_processor

gc.collect()
torch.cuda.empty_cache()

print("Qwen unloaded.")
print(
    "Allocated GPU memory:",
    round(torch.cuda.memory_allocated() / 1024**3, 2),
    "GB"
)

Qwen unloaded.
Allocated GPU memory: 25.66 GB


In [ ]:
import os

from getpass import getpass
from huggingface_hub import login, whoami

hf_token = getpass(
    "Paste your Hugging Face token. It should start with hf_: "
)

login(
    token=hf_token,
    add_to_git_credential=True
)

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

user = whoami(token=hf_token)

print("Logged in as:", user["name"])

Paste your Hugging Face token. It should start with hf_: ··········
Logged in as: sxiayuan


In [ ]:
# ===============================
# Load patched Ideogram repo
# ===============================

%cd /content

!rm -rf /content/ideogram4-optimization

!git clone https://github.com/sxiayuan/ideogram4-optimization.git


import sys
import importlib

IDEOGRAM_SRC = "/content/ideogram4-optimization/ideogram4/src"

sys.path.insert(0, IDEOGRAM_SRC)

importlib.invalidate_caches()

from ideogram4 import (
    Ideogram4Pipeline,
    Ideogram4PipelineConfig
)

import ideogram4.pipeline_ideogram4 as pipeline

print("Loaded from:")
print(pipeline.__file__)

In [ ]:
import os

from getpass import getpass
from huggingface_hub import login, whoami

hf_token = getpass(
    "Paste your Hugging Face token. It should start with hf_: "
)

login(
    token=hf_token,
    add_to_git_credential=True
)

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

user = whoami(token=hf_token)

print("Logged in as:", user["name"])

Paste your Hugging Face token. It should start with hf_: ··········


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Logged in as: sxiayuan


In [ ]:
import os
import sys
import torch

if "ideogram4" in sys.modules:
    del sys.modules["ideogram4"]

sys.path = [
    path
    for path in sys.path
    if path != "/content"
]

sys.path.insert(
    0,
    "/content/ideogram4-optimization/ideogram4/src"
)

from ideogram4 import (
    Ideogram4Pipeline,
    Ideogram4PipelineConfig
)

print("Ideogram import successful.")


device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ============================
# Load Ideogram 4 FP8
# ============================

pipe = Ideogram4Pipeline.from_pretrained(
    config=Ideogram4PipelineConfig(
        weights_repo="ideogram-ai/ideogram-4-fp8"
    ),
    device=device,
    dtype=torch.bfloat16
)


print("Ideogram FP8 loaded.")
print("Device:", device)

# ==========================================
# Inference optimizations
# ==========================================

if device == "cuda":

    torch.set_float32_matmul_precision("high")

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # Prefer memory-efficient attention
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)

print("Optimization setup complete.")



Ideogram import successful.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 9289.79 MB. The target location /content/drive/MyDrive/huggingface_cache/models--ideogram-ai--ideogram-4-fp8/blobs only has 3076.44 MB free disk space.
  warnings.warn(


transformer/diffusion_pytorch_model.safe(…): reconstructing file:   0%|          |  0.00B / 9.29GB            

transformer/diffusion_pytorch_model.safe(…): downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 9289.79 MB. The target location /content/drive/MyDrive/huggingface_cache/models--ideogram-ai--ideogram-4-fp8/blobs only has 3076.41 MB free disk space.
  warnings.warn(


...

unconditional_transformer/diffusion_pyto(…): downloading bytes:           |  0.00B            

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:744: UserWarning: Not enough free disk space to download the file. The expected file size is: 8779.28 MB. The target location /content/drive/MyDrive/huggingface_cache/models--ideogram-ai--ideogram-4-fp8/blobs only has 3076.27 MB free disk space.
  warnings.warn(


text_encoder/model.safetensors: reconstructing file:   0%|          |  0.00B / 8.78GB            

text_encoder/model.safetensors: downloading bytes:           |  0.00B            

Ideogram FP8 loaded.
Device: cuda
Optimization setup complete.


In [ ]:
import time
import torch

profile = {}


def add_time(category, elapsed):
    if category not in profile:
        profile[category] = 0
    profile[category] += elapsed


def wrap_module(name, module):

    original_forward = module.forward

    def wrapped_forward(*args, **kwargs):

        torch.cuda.synchronize()
        start = time.perf_counter()

        output = original_forward(*args, **kwargs)

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        add_time(name, elapsed)

        return output

    module.forward = wrapped_forward


# Profile important components
for name, module in pipe.conditional_transformer.named_modules():

    lower = name.lower()

    if (
        "attention" in lower
        or "mlp" in lower
        or "ffn" in lower
        or "proj" in lower
        or "norm" in lower
        or "rotary" in lower
    ):
        wrap_module(name, module)


print("Transformer profiling installed")

Transformer profiling installed


In [ ]:
if device == "cuda":

    torch.set_float32_matmul_precision("high")

    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    # Prefer memory-efficient attention
    torch.backends.cuda.enable_flash_sdp(False)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(True)

print("Optimization setup complete.")

Optimization setup complete.


In [ ]:
import json

def normalize_hex_colors(data):
    if isinstance(data, dict):
        return {
            key: normalize_hex_colors(value)
            for key, value in data.items()
        }

    if isinstance(data, list):
        return [
            normalize_hex_colors(value)
            for value in data
        ]

    if (
        isinstance(data, str)
        and len(data) == 7
        and data.startswith("#")
    ):
        return data.upper()

    return data


enhanced_caption = normalize_hex_colors(
    enhanced_caption
)

validate_ideogram_json(
    enhanced_caption
)

ideogram_prompt = json.dumps(
    enhanced_caption,
    separators=(",", ":"),
    ensure_ascii=False
)

print("JSON converted into ideogram_prompt.")
print(ideogram_prompt)

JSON converted into ideogram_prompt.
{"high_level_description":"A dramatic and intense scene of Godzilla attacking Tokyo, capturing the chaos and destruction with a cinematic and immersive perspective. The iconic monster dominates the frame, its massive form casting long shadows across the devastated cityscape. The urban environment is filled with debris and damaged buildings, emphasizing the scale of the devastation. The sky is filled with smoke and dust, adding to the apocalyptic atmosphere. The camera follows Godzilla's movements, providing a dynamic and thrilling experience. The color palette features deep, muted tones of gray and brown, punctuated by the bright red of the monster's eyes and the fiery orange of explosions. The overall composition is designed to evoke a sense of awe and terror, immersing the viewer in the heart of the battle.","style_description":{"aesthetics":"A cinematic and immersive visual style, featuring dramatic lighting, detailed textures, and a high level o

In [ ]:
import json
import re
from math import gcd, floor, lcm


def extract_size_request(user_request):
    """
    Extract either:
    1. explicit width and height
    2. aspect_ratio like 16:9, 9:16, 20:2, etc.

    Returns:
    {
        "width": int or None,
        "height": int or None,
        "aspect_ratio": str or None
    }
    """

    result = {
        "width": None,
        "height": None,
        "aspect_ratio": None
    }

    # Try JSON first
    try:
        parsed = json.loads(user_request)

        if isinstance(parsed, dict):
            # top-level width/height
            if "width" in parsed and "height" in parsed:
                result["width"] = int(parsed["width"])
                result["height"] = int(parsed["height"])
                return result

            # top-level aspect_ratio
            if "aspect_ratio" in parsed:
                result["aspect_ratio"] = str(parsed["aspect_ratio"]).replace(" ", "")
                return result

            # nested frame object
            frame = parsed.get("frame")

            if isinstance(frame, dict):
                if "width" in frame and "height" in frame:
                    result["width"] = int(frame["width"])
                    result["height"] = int(frame["height"])
                    return result

                if "aspect_ratio" in frame:
                    result["aspect_ratio"] = str(frame["aspect_ratio"]).replace(" ", "")
                    return result

    except Exception:
        pass

    # Plain-text explicit dimensions, e.g. 1344x768 or 1344 x 768
    dim_match = re.search(
        r"\b(\d{2,5})\s*[xX×]\s*(\d{2,5})\b",
        user_request
    )

    if dim_match:
        result["width"] = int(dim_match.group(1))
        result["height"] = int(dim_match.group(2))
        return result

    # Plain-text ratio, e.g. 16:9, 9:16, 20:2
    ratio_match = re.search(
        r"\b(\d+\s*:\s*\d+)\b",
        user_request
    )

    if ratio_match:
        result["aspect_ratio"] = ratio_match.group(1).replace(" ", "")
        return result

    return result


def round_to_multiple(value, multiple=64):
    return max(multiple, round(value / multiple) * multiple)


def validate_explicit_dimensions(width, height, multiple=64):
    """
    Ideogram/diffusion pipelines usually require dimensions divisible by a fixed multiple.
    This function rounds user dimensions to the nearest valid multiple.
    """

    width = round_to_multiple(width, multiple)
    height = round_to_multiple(height, multiple)

    if width <= 0 or height <= 0:
        raise ValueError("Width and height must be positive.")

    return width, height


def aspect_ratio_to_dimensions(
    aspect_ratio,
    max_side=1344,
    multiple=64,
    default_width=1024,
    default_height=1344
):
    """
    Convert any aspect ratio into valid width and height.
    Works for 16:9, 9:16, 1:1, 20:2, 10:15, etc.
    """

    if aspect_ratio is None:
        return default_width, default_height

    if ":" not in aspect_ratio:
        raise ValueError(f"Invalid aspect ratio: {aspect_ratio}")

    w_ratio, h_ratio = aspect_ratio.split(":")
    w_ratio = int(w_ratio)
    h_ratio = int(h_ratio)

    if w_ratio <= 0 or h_ratio <= 0:
        raise ValueError(f"Invalid aspect ratio: {aspect_ratio}")

    ratio_gcd = gcd(w_ratio, h_ratio)
    w_ratio = w_ratio // ratio_gcd
    h_ratio = h_ratio // ratio_gcd

    k_w = multiple // gcd(w_ratio, multiple)
    k_h = multiple // gcd(h_ratio, multiple)
    k_base = lcm(k_w, k_h)

    max_ratio_side = max(w_ratio, h_ratio)
    scale_count = floor(max_side / (max_ratio_side * k_base))

    if scale_count < 1:
        raise ValueError(
            f"Aspect ratio {aspect_ratio} is too extreme for max_side={max_side} "
            f"and multiple={multiple}."
        )

    k = k_base * scale_count

    width = w_ratio * k
    height = h_ratio * k

    return width, height


def get_ideogram_width_height(user_request):
    size_request = extract_size_request(user_request)

    requested_width = size_request["width"]
    requested_height = size_request["height"]
    requested_aspect_ratio = size_request["aspect_ratio"]

    if requested_width is not None and requested_height is not None:
        width, height = validate_explicit_dimensions(
            requested_width,
            requested_height,
            multiple=64
        )

        return width, height, f"{requested_width}x{requested_height}"

    width, height = aspect_ratio_to_dimensions(
        requested_aspect_ratio,
        max_side=1344,
        multiple=64
    )

    return width, height, requested_aspect_ratio


WIDTH, HEIGHT, REQUESTED_SIZE = get_ideogram_width_height(USER_REQUEST)

print("Requested size/aspect ratio:", REQUESTED_SIZE)
print("Ideogram width:", WIDTH)
print("Ideogram height:", HEIGHT)
print("Actual ratio:", round(WIDTH / HEIGHT, 4))

Requested size/aspect ratio: None
Ideogram width: 1024
Ideogram height: 1344
Actual ratio: 0.7619


In [ ]:
ideogram_caption = {
    "high_level_description": enhanced_caption["high_level_description"],
    "style_description": enhanced_caption["style_description"],
    "compositional_deconstruction": enhanced_caption["compositional_deconstruction"]
}

ideogram_prompt = json.dumps(
    ideogram_caption,
    separators=(",", ":"),
    ensure_ascii=False
)

print("Ideogram-compatible prompt:")
print(ideogram_prompt)
print("Prompt characters:", len(ideogram_prompt))

Ideogram-compatible prompt:
{"high_level_description":"A dramatic and intense scene of Godzilla attacking Tokyo, capturing the chaos and destruction with a cinematic and immersive perspective. The iconic monster dominates the frame, its massive form casting long shadows across the devastated cityscape. The urban environment is filled with debris and damaged buildings, emphasizing the scale of the devastation. The sky is filled with smoke and dust, adding to the apocalyptic atmosphere. The camera follows Godzilla's movements, providing a dynamic and thrilling experience. The color palette features deep, muted tones of gray and brown, punctuated by the bright red of the monster's eyes and the fiery orange of explosions. The overall composition is designed to evoke a sense of awe and terror, immersing the viewer in the heart of the battle.","style_description":{"aesthetics":"A cinematic and immersive visual style, featuring dramatic lighting, detailed textures, and a high level of realism

In [ ]:
import json

def reorder_for_ideogram(data):
    style = data["style_description"]

    if "photo" in style:
        reordered_style = {
            "aesthetics": style["aesthetics"],
            "lighting": style["lighting"],
            "photo": style["photo"],
            "medium": style["medium"],
            "color_palette": style["color_palette"]
        }
    elif "art_style" in style:
        reordered_style = {
            "aesthetics": style["aesthetics"],
            "lighting": style["lighting"],
            "medium": style["medium"],
            "art_style": style["art_style"],
            "color_palette": style["color_palette"]
        }
    else:
        raise ValueError("style_description must contain either photo or art_style.")

    comp = data["compositional_deconstruction"]

    reordered_elements = []

    for element in comp["elements"]:
        reordered_elements.append({
            "type": element["type"],
            "bbox": element["bbox"],
            "desc": element["desc"],
            "color_palette": element["color_palette"]
        })

    return {
        "high_level_description": data["high_level_description"],
        "style_description": reordered_style,
        "compositional_deconstruction": {
            "background": comp["background"],
            "elements": reordered_elements
        }
    }


enhanced_caption = reorder_for_ideogram(enhanced_caption)

ideogram_prompt = json.dumps(
    enhanced_caption,
    separators=(",", ":"),
    ensure_ascii=False
)

print("Reordered JSON prompt:")
print(json.dumps(enhanced_caption, indent=2, ensure_ascii=False))

Reordered JSON prompt:
{
  "high_level_description": "A dramatic and intense scene of Godzilla attacking Tokyo, capturing the chaos and destruction with a cinematic and immersive perspective. The iconic monster dominates the frame, its massive form casting long shadows across the devastated cityscape. The urban environment is filled with debris and damaged buildings, emphasizing the scale of the devastation. The sky is filled with smoke and dust, adding to the apocalyptic atmosphere. The camera follows Godzilla's movements, providing a dynamic and thrilling experience. The color palette features deep, muted tones of gray and brown, punctuated by the bright red of the monster's eyes and the fiery orange of explosions. The overall composition is designed to evoke a sense of awe and terror, immersing the viewer in the heart of the battle.",
  "style_description": {
    "aesthetics": "A cinematic and immersive visual style, featuring dramatic lighting, detailed textures, and a high level o

In [ ]:
import time
import torch

component_times = {}

def timed_call(name, module):

    original_forward = module.forward

    def wrapper(*args, **kwargs):

        torch.cuda.synchronize()
        start = time.perf_counter()

        result = original_forward(*args, **kwargs)

        torch.cuda.synchronize()
        elapsed = time.perf_counter() - start

        component_times[name] = (
            component_times.get(name, 0) + elapsed
        )

        return result

    module.forward = wrapper


for name in [
    "conditional_transformer",
    "unconditional_transformer",
    "text_encoder",
    "autoencoder"
]:
    if hasattr(pipe, name):
        timed_call(
            name,
            getattr(pipe, name)
        )

print("Component profiler installed")

Component profiler installed


In [ ]:
from pathlib import Path
from PIL import Image
import time
import secrets

OUTPUT_DIRECTORY = Path("/content/ideogram_outputs/draft_generations")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

SEED = secrets.randbelow(2_147_483_647)

DRAFT_WIDTH = 768
DRAFT_HEIGHT = 1024

OUTPUT_PATH = (
    OUTPUT_DIRECTORY
    / f"draft_seed_{SEED}_{DRAFT_WIDTH}x{DRAFT_HEIGHT}.png"
)

print("Generating draft with seed:", SEED)
print("Draft size:", DRAFT_WIDTH, "x", DRAFT_HEIGHT)

start_time = time.perf_counter()

with torch.inference_mode():

  result = pipe(
    ideogram_prompt,
    width=DRAFT_WIDTH,
    height=DRAFT_HEIGHT,
    num_steps=12,
    guidance_schedule=(4.0,) + (8.0,) * 11,
    mu=0.55,
    std=1.75,
    seed=SEED,
    use_batched_cfg=True,
    profile_cfg=True
  )

generation_time = time.perf_counter() - start_time

if isinstance(result, (list, tuple)):
    generated_image = result[0]
elif hasattr(result, "images"):
    generated_image = result.images[0]
else:
    generated_image = result

generated_image.save(OUTPUT_PATH, format="PNG")

print("Saved:", OUTPUT_PATH)
print("Generation time:", round(generation_time, 2), "seconds")
print("Resolution:", generated_image.size)
print("Generation time:", round(generation_time, 2), "seconds")

In [ ]:
for k,v in component_times.items():
    print(k, f"{v:.3f}s")

In [ ]:
from collections import defaultdict

summary = defaultdict(float)

for name, t in profile.items():

    if "attention" in name.lower():
        summary["Attention"] += t

    elif "mlp" in name.lower() or "ffn" in name.lower():
        summary["MLP/FFN"] += t

    elif "proj" in name.lower():
        summary["Projection"] += t

    elif "norm" in name.lower():
        summary["Normalization"] += t

    elif "rotary" in name.lower():
        summary["RoPE"] += t

    else:
        summary["Other"] += t


for k, v in sorted(
    summary.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(
        f"{k}: {v:.3f}s"
    )

In [ ]:
from PIL import Image
from IPython.display import display

SEED = 532938188

path = (
    Path("/content/ideogram_outputs/draft_generations")
    / f"draft_seed_{SEED}_768x1024.png"
)

display(Image.open(path))